In [1]:
# ======================
# CONFIGURATION DU CHEMIN DU PROJET 
# ======================
import sys
import os

# Remonter à la racine du projet (depuis notebooks/training/)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"✅ Racine du projet ajoutée au PYTHONPATH : {project_root}")
print(f"   sys.path : {sys.path[0]}")

✅ Racine du projet ajoutée au PYTHONPATH : c:\Users\SOL\Downloads\sentiment_analysis_project
   sys.path : c:\Users\SOL\Downloads\sentiment_analysis_project


## importation de model et base de donnée

In [2]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification, get_linear_schedule_with_warmup
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch

# 1. Tokenizer RoBERTa
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

# 2. Dataset adapté pour Transformers (TEXTE BRUT)
class SentimentRobertaDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # ⚠️ TEXTE BRUT ORIGINAL (pas df['text_clean'] !)
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

# 3. Créer datasets avec TEXTE BRUT (vérifiez que train_df/test_df ont 'text')
from sklearn.model_selection import train_test_split
import pandas as pd
test_size = 0.2
random_state = 42
df=pd.read_csv(r'C:\Users\SOL\Downloads\sentiment_analysis_project\data\sentiment_dataset.csv')

train_df, test_df = train_test_split(
        df, 
        test_size=test_size, 
        random_state=random_state, 
        stratify=df['label']  # Préserver la distribution des classes
    )
print("📊 Création des datasets RoBERTa avec TEXTE BRUT...")
train_dataset = SentimentRobertaDataset(
    texts=train_df['text'].tolist(),   # ← TEXTE ORIGINAL SANS PRÉTRAITEMENT
    labels=train_df['label'].tolist(),
    tokenizer=tokenizer,
    max_length=128
)

test_dataset = SentimentRobertaDataset(
    texts=test_df['text'].tolist(),    # ← TEXTE ORIGINAL SANS PRÉTRAITEMENT
    labels=test_df['label'].tolist(),
    tokenizer=tokenizer,
    max_length=128
)

train_loader_roberta = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
test_loader_roberta = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"✅ Datasets créés :")
print(f"   - Train : {len(train_dataset)} échantillons")
print(f"   - Test  : {len(test_dataset)} échantillons")

# 4. Modèle pré-entraîné
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = RobertaForSequenceClassification.from_pretrained(
    'roberta-base',
    num_labels=3,
    problem_type="single_label_classification"
).to(device)

print(f"\n🧠 Modèle RoBERTa-base chargé ({sum(p.numel() for p in model.parameters())/1e6:.1f}M paramètres)")
print(f"   → Utilisation du texte BRUT (pas de prétraitement custom)")
print(f"   → Emojis, '!!!', majuscules préservés comme signaux émotionnels")

c:\Users\SOL\anaconda3\envs\deeplearning\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


📊 Création des datasets RoBERTa avec TEXTE BRUT...
✅ Datasets créés :
   - Train : 24985 échantillons
   - Test  : 6247 échantillons


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 995.23it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]              
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside


🧠 Modèle RoBERTa-base chargé (124.6M paramètres)
   → Utilisation du texte BRUT (pas de prétraitement custom)
   → Emojis, '!!!', majuscules préservés comme signaux émotionnels


## FINE-TUNING

In [4]:
# ROBERTA - FINE-TUNING 
from torch.optim import AdamW  
from sklearn.metrics import f1_score, accuracy_score, classification_report
import time
from src.trainer import train_roberta

# 1. Optimiseur + Scheduler
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)

total_steps = len(train_loader_roberta) * 4
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

# 2. Entraînement (4 epochs)

history, model_path = train_roberta(
    model=model,
    train_loader=train_loader_roberta,
    val_loader=test_loader_roberta,
    optimizer=optimizer,
    scheduler=scheduler,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    num_epochs=4,
    patience=2,
    save_path=r'C:\Users\SOL\Downloads\sentiment_analysis_project\models\trinary',
    model_name='RoBERTa-base',
    verbose=True  
)




DÉBUT DU FINE-TUNING - RoBERTa-base
⚙️  Configuration : LR=2e-05 | Batch=32 | Epochs=4
⚠️  Texte BRUT utilisé (pas de prétraitement custom)

Epoch  1/4 | Train Loss: 0.6470 | Train F1: 0.7234 | Val Loss: 0.5640 | Val F1 (macro): 0.7644 | Val Acc: 0.7613 | Time: 325s
   → ✅ Meilleur modèle sauvegardé (F1 macro: 0.7644)
Epoch  2/4 | Train Loss: 0.5048 | Train F1: 0.7966 | Val Loss: 0.5546 | Val F1 (macro): 0.7730 | Val Acc: 0.7704 | Time: 326s
   → ✅ Meilleur modèle sauvegardé (F1 macro: 0.7730)
Epoch  3/4 | Train Loss: 0.4119 | Train F1: 0.8363 | Val Loss: 0.6024 | Val F1 (macro): 0.7719 | Val Acc: 0.7700 | Time: 326s
   → ⏳ Patience: 1/2
Epoch  4/4 | Train Loss: 0.3434 | Train F1: 0.8703 | Val Loss: 0.6236 | Val F1 (macro): 0.7716 | Val Acc: 0.7688 | Time: 326s
   → ⏳ Patience: 2/2

🛑 Early stopping déclenché à l'epoch 4

RÉSULTATS FINAUX - RoBERTa-base
F1 Macro      : 0.7730
Meilleur modèle sauvegardé : 'C:\Users\SOL\Downloads\sentiment_analysis_project\models\trinary'



## evaluation

In [5]:
# 7. Évaluation finale (utilisez votre fonction evaluate_model existante)
from src.evaluate import evaluate_model

results = evaluate_model(
    model=model,
    dataloader=test_loader_roberta,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    class_names=['Négatif', 'Neutre', 'Positif'],
    save_dir=r'C:\Users\SOL\Downloads\sentiment_analysis_project\reports\figures',
    model_name='roberta_base')


📊 ÉVALUATION DU MODÈLE : roberta_base

Accuracy      : 0.7688
F1 Macro      : 0.7716
F1 Weighted   : 0.7688
ROC AUC Micro : 0.9184
ROC AUC Macro : 0.9150

Classification Report:
              precision    recall  f1-score   support

     Négatif     0.7697    0.7930    0.7812      1821
      Neutre     0.7129    0.7034    0.7081      2330
     Positif     0.8301    0.8206    0.8253      2096

    accuracy                         0.7688      6247
   macro avg     0.7709    0.7723    0.7716      6247
weighted avg     0.7688    0.7688    0.7688      6247

   → Matrice de confusion sauvegardée : confusion_matrix.png/csv
   → Courbes ROC sauvegardées : roc_curves.png

✅ Évaluation terminée - Résultats sauvegardés dans : C:\Users\SOL\Downloads\sentiment_analysis_project\reports\figures\roberta_base
